# Transformers 模型量化技术：AWQ（OPT-2.7B）

![img](https://huggingface.co/datasets/ybelkada/documentation-images/resolve/main/Thumbnail.png)

在2023年6月，Ji Lin等人发表了论文 [AWQ：Activation-aware Weight Quantization for LLM Compression and Acceleration](https://arxiv.org/pdf/2306.00978.pdf)。

这篇论文详细介绍了一种激活感知权重量化算法，可以用于压缩任何基于 Transformer 的语言模型，同时只有微小的性能下降。关于 AWQ 算法的详细介绍，见[MIT Han Song 教授分享](https://hanlab.mit.edu/projects/awq)。

transformers 现在支持两个不同的 AWQ 开源实现库：

- [AutoAWQ](https://github.com/casper-hansen/AutoAWQ)
- [LLM-AWQ](https://github.com/mit-han-lab/llm-awq) 


因为 LLM-AWQ 不支持 Nvidia T4 GPU（课程演示 GPU），所以我们使用 AutoAWQ 库来介绍和演示 AWQ 模型量化技术。

## 使用 AutoAWQ 量化模型

下面我们以 `facebook opt-2.7B` 模型为例，使用 `AutoAWQ` 库实现的 AWQ 算法实现模型量化。

In [24]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model_name_or_path = "facebook/opt-1.3b"
quant_model_dir = "models/opt-1.3b-awq"

quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

In [46]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(model_name_or_path, device_map="cuda").to(0)
# 获取当前模型占用的 GPU显存（差值为预留给 PyTorch 的显存）
memory_footprint_bytes = model.get_memory_footprint()
memory_footprint_mib = memory_footprint_bytes / (1024 ** 2)  # 转换为 MiB

print(f"{memory_footprint_mib:.2f}MiB")

5019.22MiB


In [23]:
from huggingface_hub import hf_hub_download, snapshot_download
# 下载整个仓库
snapshot_download(repo_id="facebook/opt-1.3b")

Error while downloading from https://cdn-lfs.hf.co/repos/07/3d/073de108a2c59896a27d14fab4481eb23b2158f96739f10e132b57dd7e2f23fe/ec9f807bae6c0cf0fb1ca03119056279f6446655d64d19d7e46ff1f3657b2fd9?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27tf_model.h5%3B+filename%3D%22tf_model.h5%22%3B&Expires=1742642566&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MjY0MjU2Nn19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy8wNy8zZC8wNzNkZTEwOGEyYzU5ODk2YTI3ZDE0ZmFiNDQ4MWViMjNiMjE1OGY5NjczOWYxMGUxMzJiNTdkZDdlMmYyM2ZlL2VjOWY4MDdiYWU2YzBjZjBmYjFjYTAzMTE5MDU2Mjc5ZjY0NDY2NTVkNjRkMTlkN2U0NmZmMWYzNjU3YjJmZDk%7EcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIn1dfQ__&Signature=Rm4K1VKOkRc6PuxWgzqGtWn3g9aFRFinVKSBl5NB80IkQQ99CbV%7EhhN4J8CiH-3cjHfBDOHtQGM1gqP5Hn9spnR5nircCG3vCGM2HwUJR404WvsmJGl6IKxGcNUmk1CtC028bdcZHXMUnLhH8dVv7EEuSkjxwbogkfd5BxPzk5%7EnxIM9ks3mYae4OEdu1CNfM9UuTN4V69KmnlsR2VjDXNI2Y3agzHjobjlqJa4xv8ThUfCXIyX9jDDiDrIF6dR4-0K84GwsQxuwI5Ngwvo

'/root/.cache/huggingface/hub/models--facebook--opt-1.3b/snapshots/3f5c25d0bc631cb57ac65913f76e22c2dfb61d62'

In [47]:
# 加载模型
model = AutoAWQForCausalLM.from_pretrained(model_name_or_path, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True)

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 96791.63it/s]


In [26]:
# 量化模型
model.quantize(tokenizer, quant_config=quant_config)

Repo card metadata block was not found. Setting CardData to empty.
AWQ: 100%|██████████| 24/24 [08:38<00:00, 21.62s/it]


### 实测 AWQ 量化模型：GPU显存占用峰值超过10GB


```shell
Sun Dec 24 15:21:35 2023
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:0D.0 Off |                    0 |
| N/A   53C    P0              71W /  70W |   7261MiB / 15360MiB |     97%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|```

In [27]:
quant_config

{'zero_point': True, 'q_group_size': 128, 'w_bit': 4, 'version': 'GEMM'}

#### Transformers 兼容性配置

为了使`quant_config` 与 transformers 兼容，我们需要修改配置文件：`使用 Transformers.AwqConfig 来实例化量化模型配置`

In [28]:
from transformers import AwqConfig, AutoConfig

# 修改配置文件以使其与transformers集成兼容
quantization_config = AwqConfig(
    bits=quant_config["w_bit"],
    group_size=quant_config["q_group_size"],
    zero_point=quant_config["zero_point"],
    version=quant_config["version"].lower(),
).to_dict()

# 预训练的transformers模型存储在model属性中，我们需要传递一个字典
model.model.config.quantization_config = quantization_config

In [29]:
# 保存模型权重
model.save_quantized(quant_model_dir)
# 保存分词器
tokenizer.save_pretrained(quant_model_dir)  

('models/opt-1.3b-awq/tokenizer_config.json',
 'models/opt-1.3b-awq/special_tokens_map.json',
 'models/opt-1.3b-awq/vocab.json',
 'models/opt-1.3b-awq/merges.txt',
 'models/opt-1.3b-awq/added_tokens.json',
 'models/opt-1.3b-awq/tokenizer.json')

In [30]:
model.eval()

OptAWQForCausalLM(
  (model): OPTForCausalLM(
    (model): OPTModel(
      (decoder): OPTDecoder(
        (embed_tokens): Embedding(50272, 2048, padding_idx=1)
        (embed_positions): OPTLearnedPositionalEmbedding(2050, 2048)
        (final_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (layers): ModuleList(
          (0-23): 24 x OPTDecoderLayer(
            (self_attn): OPTAttention(
              (k_proj): WQLinear_GEMM(in_features=2048, out_features=2048, bias=True, w_bit=4, group_size=128)
              (v_proj): WQLinear_GEMM(in_features=2048, out_features=2048, bias=True, w_bit=4, group_size=128)
              (q_proj): WQLinear_GEMM(in_features=2048, out_features=2048, bias=True, w_bit=4, group_size=128)
              (out_proj): WQLinear_GEMM(in_features=2048, out_features=2048, bias=True, w_bit=4, group_size=128)
            )
            (activation_fn): ReLU()
            (self_attn_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affin

### 使用 GPU 加载量化模型

In [31]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(quant_model_dir)
model = AutoModelForCausalLM.from_pretrained(quant_model_dir, device_map="cuda").to(0)

In [35]:
def generate_text(text):
    inputs = tokenizer(text, return_tensors="pt").to(0)

    out = model.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(out[0], skip_special_tokens=True)


In [36]:
result = generate_text("Merry Christmas! I'm glad to")
print(result)

Merry Christmas! I'm glad to your  I'm I am. I'm I am. I am. I am I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am. I am.


In [37]:
result = generate_text("The woman worked as a")
print(result)

The woman worked as a as as the as the the the the the the the the the the the..
Hhhhhhhhhhhhhhhh hhhhhhh


In [39]:
from transformers import pipeline


# 使用 GPU 加载原始的 OPT-125m 模型
generator = pipeline('text-generation',
                     model=quant_model_dir,
                     device=0,
                     do_sample=True,
                     num_return_sequences=3)

You have loaded an AWQ model on CPU and have a CUDA device available, make sure to set your model on a GPU device in order to run your model.


In [40]:
generator("The woman worked as a")

[{'generated_text': 'The woman worked as a the the the the the the the the the the the the the the the'},
 {'generated_text': 'The woman worked as a the the the the the the the the the the the the the the the'},
 {'generated_text': "The woman worked as a  the man man  The woman\nIt's a her own man h"}]

In [41]:
generator("The man worked as a")

[{'generated_text': 'The man worked as a a guy.  The man.\nThe mana.\nAs a'},
 {'generated_text': 'The man worked as a the the the the the the the the the the...\nYou should you'},
 {'generated_text': 'The man worked as a man man man man\n\n1 2 2 2 2 2\n\nThe'}]

In [42]:
# 获取当前模型占用的 GPU显存（差值为预留给 PyTorch 的显存）
memory_footprint_bytes = model.get_memory_footprint()
memory_footprint_mib = memory_footprint_bytes / (1024 ** 2)  # 转换为 MiB

print(f"{memory_footprint_mib:.2f}MiB")

804.11MiB


In [48]:
804.11/5019.22

0.16020616749216013